In [1]:
import pandas as pd
from pathlib import Path

file_path = "../data/munchee_optimization_dataset.xlsx"

xls = pd.ExcelFile(file_path)
print(xls.sheet_names)

['README', 'Depot', 'Vehicles', 'Routes', 'Route_Schedule', 'Shops', 'Products', 'Inventory', 'Weekly_Demand', 'Shop_Summary', 'Order_History_8W', 'Distance_Matrix']


In [2]:
products_df = pd.read_excel(file_path, sheet_name="Products")
inventory_df = pd.read_excel(file_path, sheet_name="Inventory")
demand_df = pd.read_excel(file_path, sheet_name="Weekly_Demand")
shops_df = pd.read_excel(file_path, sheet_name="Shops")
vehicles_df = pd.read_excel(file_path, sheet_name="Vehicles")
routes_df = pd.read_excel(file_path, sheet_name="Routes")
distance_df = pd.read_excel(file_path, sheet_name="Distance_Matrix")
depot_df = pd.read_excel(file_path, sheet_name="Depot")

In [3]:
for df in [products_df, inventory_df, demand_df, shops_df, vehicles_df, routes_df, distance_df, depot_df]:
    df.columns = df.columns.str.strip().str.lower()

In [4]:
print("Products:", products_df.shape)
print("Inventory:", inventory_df.shape)
print("Demand:", demand_df.shape)
print("Shops:", shops_df.shape)
print("Vehicles:", vehicles_df.shape)
print("Routes:", routes_df.shape)
print("Distance:", distance_df.shape)

print(products_df.head())
print(inventory_df.head())
print(demand_df.head())

Products: (10, 9)
Inventory: (10, 5)
Demand: (500, 4)
Shops: (50, 14)
Vehicles: (2, 9)
Routes: (5, 12)
Distance: (2601, 6)
  product_id             product_name  packs_per_carton  carton_weight_kg  \
0        P01             Munchee Nice                24               1.8   
1        P02            Munchee Marie                24               1.9   
2        P03  Munchee Milk Short Cake                20               2.1   
3        P04  Munchee Chocolate Cream                18               2.2   
4        P05       Munchee Lemon Puff                18               2.2   

   carton_volume_m3  wholesale_price_lkr  recommended_sellout_lkr  \
0             0.030                 1720                     2100   
1             0.031                 1750                     2140   
2             0.034                 2050                     2520   
3             0.036                 2380                     2920   
4             0.036                 2320                     2860   


In [5]:
products = products_df["product_id"].tolist()
shops = shops_df["shop_id"].tolist()
vehicles = vehicles_df["vehicle_id"].tolist()
routes = routes_df["route_id"].tolist()

print("No. of products:", len(products))
print("No. of shops:", len(shops))
print("No. of vehicles:", len(vehicles))
print("No. of routes:", len(routes))

No. of products: 10
No. of shops: 50
No. of vehicles: 2
No. of routes: 5


In [6]:
stock = dict(zip(inventory_df["product_id"], inventory_df["opening_stock_cartons"]))

In [7]:
safety_stock = dict(zip(inventory_df["product_id"], inventory_df["safety_stock_cartons"]))

In [8]:
usable_stock = {
    p: max(stock[p] - safety_stock[p], 0)
    for p in products
}

In [9]:
weights = dict(zip(products_df["product_id"], products_df["carton_weight_kg"]))

In [10]:
volumes = dict(zip(products_df["product_id"], products_df["carton_volume_m3"]))

In [11]:
margin = dict(zip(products_df["product_id"], products_df["gross_margin_lkr"]))

In [12]:
vehicle_carton_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_cartons"]))

In [13]:
vehicle_weight_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_payload_kg"]))

In [14]:
vehicle_volume_capacity = dict(zip(vehicles_df["vehicle_id"], vehicles_df["max_volume_m3"]))

In [15]:
demand = {}
for _, row in demand_df.iterrows():
    demand[(row["shop_id"], row["product_id"])] = row["weekly_demand_cartons"]

In [16]:
priority = {}
for _, row in demand_df.iterrows():
    priority[(row["shop_id"], row["product_id"])] = row["priority_rank"]

In [17]:
shop_to_route = dict(zip(shops_df["shop_id"], shops_df["route_id"]))

In [18]:
distance_lookup = {}
time_lookup = {}

for _, row in distance_df.iterrows():
    distance_lookup[(row["from_node"], row["to_node"])] = row["estimated_distance_km"]
    time_lookup[(row["from_node"], row["to_node"])] = row["estimated_travel_time_min"]